In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\usability_predictors.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data whic

In [2]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

data_tens = torch.tensor(data.values, dtype=torch.float32)

In [ ]:
StandardEvaluations: List[EvaluationFunction] = [
    UsabilityEvaluator(),
    # AeroEvaluator(),
    # ErgonomicsEvaluator(),
    # AestheticsEvaluator(mode="Embedding", batch_size=64),
    # StructuralEvaluator(),
    # ValidationEvaluator(),
    # FrameValidityEvaluator()
]

evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1


In [4]:
num_data = data.shape[0]
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")
image_embeddings = conditioning.sample_image_embedding(num_data, split="test")
condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Embedding": image_embeddings}
# condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [5]:
#calcualte gradient of scores wrt data_tens

data_tens.requires_grad = True
eval_scores = evaluator(data_tens, condition)
score_sum = eval_scores.sum()
score_sum.backward()


In [6]:
#check for infs and nans in data_tens
if torch.any(torch.isnan(data_tens)) or torch.any(torch.isinf(data_tens)):
    print("Data tensor contains NaN or Inf values.")

In [7]:
data_tens[1861]

tensor([  0.0000,   0.0000, 544.4359,  80.0000, 131.2000,  45.0000, 553.8000,
        100.0000, 792.5038,  45.0000,  40.0000,  28.6000,  36.5000,  23.3500,
         13.0000,  15.0000,   9.0000,  35.0000,  53.8000,  44.3000, 330.0000,
        350.0000,  16.0000,  18.0000,   0.0000,   0.0000, 130.0000,   2.0000,
          0.9000,   1.1000,   0.9000,   1.2000,   1.0000,   0.9000, 736.0000,
        114.0000, 736.0000, 114.0000,   0.0000,  68.0000,  45.0000, 692.0000,
         31.8000,   0.0000,   0.0000,   1.0000,   1.0000,   1.0000,   8.0000,
          0.0000,   0.0000, 204.0000, 204.0000, 204.0000,   1.0000,   1.0000,
         80.0000,  80.0000, 278.0000, 718.0000,  34.9000, 150.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   1.0000,   0.0000,   0.0000,   0.0000,
          1.0000,   0.0000,   0.0000,   1.0000,   0.0000,   0.0000,   0.0000,
          1.0000,   0.0000,   0.0000,   1.0000,   0.0000,   0.0000,   0.0000,
          1.0000,   1.0000,   0.0000,   0.0000,   1.0000,   0.00

In [8]:
#get nan indices of data_tens.grad
nan_indices = torch.isnan(data_tens.grad).nonzero(as_tuple=True)
print("nan indices: ", nan_indices)

nan indices:  (tensor([], dtype=torch.int64), tensor([], dtype=torch.int64))


In [9]:
isobjective = torch.tensor(requirement_types) == 1
objective_scores = eval_scores[:, isobjective].detach().numpy()
# constraint_scores = eval_scores[:, ~isobjective].detach().numpy()

In [10]:
main_scorer = construct_scorer(MainScores, StandardEvaluations, data.columns)
detailed_scorer = construct_scorer(DetailedScores, StandardEvaluations, data.columns)

IndexError: index 1 is out of bounds for axis 0 with size 1

In [ ]:
main_scorer(data_tens.detach(), condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Hypervolume                     0.000000e+00
Constraint Satisfaction Rate    8.585957e-01
Maximum Mean Discrepancy       -7.171762e-08
dtype: float64

In [ ]:
detailed_scorer(data_tens.detach(), condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Min Objective Score: Usability Score - 0 to 1                                                                 0.791411
Min Objective Score: Drag Force                                                                              27.732685
Min Objective Score: Knee Angle Error                                                                       197.293720
Min Objective Score: Hip Angle Error                                                                        805.295530
Min Objective Score: Arm Angle Error                                                                        842.159670
Min Objective Score: Cosine Similarity to Embedding                                                           0.286874
Min Objective Score: Mass                                                                                    14.987815
Min Objective Score: Planar Compliance                                                                      115.509544
Min Objective Score: Transverse Compliance      